# 04 — Explainability and error analysis (RQ5)

**RQ5.** Which behaviours drive the predictions, do model families agree on them, and
what do the sessions they get wrong look like?

| Section | Question | Source |
|---|---|---|
| 1 | How was each model explained? | `06_explain.py` runs |
| 2 | Which feature **groups** matter most — is it the cart block (H5)? | group SHAP |
| 3 | Which single features matter, per model? | feature SHAP |
| 4 | Do the model families agree? | Spearman rank agreement |
| 5 | Individual predictions (beeswarm, waterfalls) | `06_explain.py` figures |
| 6 | What characterises false positives and false negatives? | error groups |
| 7 | Does performance depend on how long the session turns out to be? | length cohorts |
| 8 | Do all models miss the **same** buyers? (incl. the GRU) | RQ1 test predictions |
| 9 | Key findings | report |

**Method.** Importance = share of mean |attribution| (TreeSHAP for RF / LightGBM,
LinearSHAP for LR, DeepSHAP for the MLP, attention masks for TabNet), with one-hot
columns summed back to their feature and features summed to their group. TabNet
attention is not SHAP: compare its *ranking*, not its magnitudes. The GRU is not
attributed, but it is included in the model-agnostic error analysis (section 8).

**Run order.** Scripts 01–03 and 06 (`python scripts/06_explain.py --mode dev --ks 3 5 10`),
then `NB_MODE=dev jupyter lab`. Figures go to `results/figures/xai_*_<mode>.png`.

In [ ]:
import json
import re
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "config" / "config.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for p in (ROOT, ROOT / "scripts"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import PercentFormatter

from _experiment import latest_runs, primary_predictions
from src.config import resolve_path
from src.features.build_features import load_feature_table
from src.utils.notebook import (INK_MUTED, PALETTE, load_result, model_colours, notebook_config,
                                save_figure, use_style)
from src.utils.results_logger import load_runs

cfg = notebook_config()
use_style()
pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_width_chars(160)

MODE = cfg["data"]["mode"]
K0 = cfg["sessionize"]["default_k"]
COLOURS = model_colours(cfg)
GROUPS = list(cfg["features"]["groups"])
GROUP_OF = {f: g for g, fs in cfg["features"]["groups"].items() for f in fs}
GROUP_COLOURS = {g: PALETTE[i] for i, g in enumerate(GROUPS)}
ERROR_GROUPS = cfg["explainability"]["error_groups"]
ERR_LABEL = {"true_positive": "TP", "false_positive": "FP", "false_negative": "FN", "true_negative": "TN"}
TABLES = resolve_path(cfg, "tables_dir")
FIGURES = resolve_path(cfg, "figures_dir")

# K values that script 06 has explained, found from its output files.
KS = sorted(int(m.group(1)) for p in TABLES.glob(f"rq5_importance_groups_k*_{MODE}.csv")
            if (m := re.search(r"_k(\d+)_", p.name)))
if KS and K0 not in KS:
    K0 = KS[0]
hyp5 = load_result(cfg, "rq5_hypothesis", "06_explain.py")


def per_k(stem: str) -> dict:
    return {k: load_result(cfg, f"{stem}_k{k}", "06_explain.py") for k in KS}


groups_imp = per_k("rq5_importance_groups")
feats_imp = per_k("rq5_importance_features")
agreement = per_k("rq5_rank_agreement")
by_error = per_k("rq5_attribution_by_error_group")
profiles = per_k("rq5_error_profile")
cohorts = per_k("rq5_metrics_by_session_length")
XAI_MODELS = [m for m in cfg["models"]["enabled"] if KS and m in groups_imp[K0].columns]


def ordered_models(models):
    return [m for m in cfg["models"]["enabled"] if m in set(models)]


summary = {"mode": MODE, "k_explained": KS}
print(f"mode={MODE}  explained K={KS}  default K={K0}  models={XAI_MODELS}")
if not KS:
    print("No RQ5 results yet — run `python scripts/06_explain.py --mode", MODE + "` first. Sections 1–7 are skipped.")

## 1. How each model was explained

In [ ]:
runs = load_runs(cfg, "rq5_explain")
if runs.height:
    runs = runs.filter(pl.col("data_mode") == MODE).sort("started_at").group_by("k", "model", maintain_order=True).last()
    display(runs.select("k", "model", pl.col("param_method").alias("method"), pl.col("param_scale").alias("scale"),
                        pl.col("param_n_explained").cast(pl.Int64).alias("sessions explained"),
                        "top_group", "top_feature").sort("k", "model"))

## 2. Feature-group importance (H5)

Share of total attribution per group G1–G5, one bar per model. **H5** predicts that the
cart-signal group (G2) ranks first for every model family.

In [ ]:
if KS:
    fig, axes = plt.subplots(1, len(KS), figsize=(5 * len(KS), 0.45 * len(XAI_MODELS) + 1.8), sharey=True)
    axes = np.atleast_1d(axes)
    for ax, k in zip(axes, KS):
        g = groups_imp[k]
        models = ordered_models([c for c in g.columns if c not in ("feature", "mean_share")])
        y = np.arange(len(models))[::-1]
        left = np.zeros(len(models))
        for grp in GROUPS:
            row = g.filter(pl.col("feature") == grp)
            vals = np.array([row[m][0] if row.height else 0.0 for m in models])
            ax.barh(y, vals, left=left, height=0.62, color=GROUP_COLOURS[grp], edgecolor="white", linewidth=2,
                    label=grp)
            for yi, l, v in zip(y, left, vals):
                if v >= 0.08:
                    ax.text(l + v / 2, yi, f"{v:.0%}", ha="center", va="center", fontsize=8, color="white")
            left += vals
        ax.set_yticks(y, models)
        ax.set_xlim(0, 1)
        ax.xaxis.set_major_formatter(PercentFormatter(1))
        ax.grid(axis="y", visible=False)
        ax.set_title(f"K={k}", fontsize=10)
    axes[-1].legend(loc="upper left", bbox_to_anchor=(1.02, 1))
    fig.suptitle("Share of attribution by feature group", x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    save_figure(fig, cfg, "xai_group_shares")
    plt.show()
if hyp5:
    h5 = hyp5["H5_cart_signals_rank_highest_for_every_family"]
    print(f"H5: {h5['verdict']} — G2 ranked first in {h5['cart_group_top']} (model, K) pairs\n   measure: {h5['measure']}")
    display(pl.DataFrame(h5["evidence"]).with_columns(pl.col("top_share", "cart_group_share").round(3)))
    summary["H5"] = {k: h5[k] for k in ("verdict", "cart_group_top")}

## 3. Feature importance

Share of attribution per feature (rows sorted by the mean over models). Darker = larger
share. Read across a row: a feature dark for every model is a robust driver; one dark
for a single model is model-specific.

In [ ]:
if KS:
    f = feats_imp[K0].sort("mean_share", descending=True).head(15)
    models = ordered_models([c for c in f.columns if c not in ("feature", "mean_share")])
    mat = f.select(models).to_numpy()
    cmap = LinearSegmentedColormap.from_list("seq", ["#f6f9fe", PALETTE[0], "#0d366b"])
    fig, ax = plt.subplots(figsize=(1.15 * len(models) + 3.5, 0.38 * f.height + 1.5))
    im = ax.imshow(mat, cmap=cmap, vmin=0, vmax=max(mat.max(), 1e-9), aspect="auto")
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            ax.text(j, i, f"{mat[i, j]:.0%}", ha="center", va="center", fontsize=8,
                    color="white" if mat[i, j] > 0.55 * mat.max() else "#0b0b0b")
    ax.set_xticks(range(len(models)), models, rotation=30, ha="right")
    ax.set_yticks(range(f.height), f["feature"])
    for tick, feat in zip(ax.get_yticklabels(), f["feature"]):
        tick.set_color(GROUP_COLOURS.get(GROUP_OF.get(feat), "#0b0b0b"))
    ax.grid(False)
    fig.colorbar(im, ax=ax, shrink=0.7, label="share of attribution", format=PercentFormatter(1))
    ax.set_title(f"Top 15 features by mean share (K={K0}; labels coloured by group)")
    save_figure(fig, cfg, "xai_feature_heatmap")
    plt.show()
    summary["top_features"] = f.head(5).select("feature", "mean_share").to_dicts()

## 4. Do the model families agree?

Spearman correlation between two models' importance rankings (1 = same order).
Group-level agreement is usually higher than feature-level agreement, because
correlated features (section 6 of notebook 02) can swap credit within a group.

In [ ]:
if KS:
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8))
    cmap = LinearSegmentedColormap.from_list("div", [PALETTE[1], "#f0efec", PALETTE[0]])
    for ax, level in zip(axes, ("feature", "group")):
        a = agreement[K0].filter(pl.col("level") == level)
        models = ordered_models(set(a["model_a"]) | set(a["model_b"]))
        idx = {m: i for i, m in enumerate(models)}
        mat = np.eye(len(models))
        for r in a.iter_rows(named=True):
            i, j = idx[r["model_a"]], idx[r["model_b"]]
            mat[i, j] = mat[j, i] = r["spearman_rho"]
        im = ax.imshow(mat, cmap=cmap, vmin=-1, vmax=1)
        for i in range(len(models)):
            for j in range(len(models)):
                if i != j:
                    ax.text(j, i, f"{mat[i, j]:.2f}", ha="center", va="center", fontsize=8)
        ax.set_xticks(range(len(models)), models, rotation=35, ha="right", fontsize=8)
        ax.set_yticks(range(len(models)), models if level == "feature" else [""] * len(models), fontsize=8)
        ax.grid(False)
        ax.set_title(f"{level}-level rank agreement (K={K0})", fontsize=10)
    fig.colorbar(im, ax=axes, shrink=0.75, label="Spearman ρ")
    save_figure(fig, cfg, "xai_rank_agreement")
    plt.show()
    agg = (pl.concat([agreement[k].with_columns(k=pl.lit(k)) for k in KS])
           .group_by("k", "level").agg(mean_rho=pl.col("spearman_rho").mean(), min_rho=pl.col("spearman_rho").min())
           .sort("k", "level"))
    display(agg)
    summary["mean_rank_agreement"] = agg.to_dicts()

## 5. Individual predictions

The beeswarm shows every explained session: position = attribution, colour = feature
value. Waterfalls break down the most confident example of each outcome. Shown for the
explained model with the best test PR-AUC at the default K (all models' figures are in
`results/figures/rq5_*`).

In [ ]:
cmp = load_result(cfg, "rq1_model_comparison", "03_run_experiments.py")
focus = None
if KS and cmp is not None:
    best = cmp.filter((pl.col("k") == K0) & pl.col("model").is_in(XAI_MODELS)).sort("pr_auc_mean", descending=True)
    focus = best["model"][0] if best.height else None
focus = focus or (XAI_MODELS[0] if XAI_MODELS else None)
summary["focus_model"] = focus

def show_images(paths, titles, ncols=2, width=14):
    paths = [p for p in paths if p.exists()]
    if not paths:
        print("(figures not found)")
        return
    nrows = int(np.ceil(len(paths) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(width, 4.2 * nrows))
    for ax in np.atleast_1d(axes).flat:
        ax.axis("off")
    for ax, p, t in zip(np.atleast_1d(axes).flat, paths, titles):
        ax.imshow(mpimg.imread(p))
        ax.set_title(t, fontsize=10)
    fig.tight_layout()
    plt.show()

if focus:
    print(f"Focus model: {focus} (K={K0})")
    show_images([FIGURES / f"rq5_beeswarm_{focus}_k{K0}_{MODE}.png", FIGURES / f"rq5_importance_{focus}_k{K0}_{MODE}.png"],
                ["beeswarm", "mean |attribution|"])
    show_images([FIGURES / f"rq5_waterfall_{focus}_k{K0}_{MODE}_{g}_1.png" for g in ERROR_GROUPS],
                [f"most confident {ERR_LABEL[g]}" for g in ERROR_GROUPS])

## 6. What the errors look like

Sessions in the explained sample, split by outcome at each model's tuned threshold.
Observable prefix traits per outcome, then the mean attribution per feature for the
focus model. *Total events* is the whole session's length — known only afterwards,
used here to describe errors, never as an input.

Expectation to check: **false positives** look like buyers (they carted) but left;
**false negatives** bought without a cart signal in the first K events.

In [ ]:
if KS:
    prof = profiles[K0]
    traits = [("mean_has_cart", "has a cart in prefix", True), ("mean_first_cart_position", "first cart position", False),
              ("mean_unique_products_count", "distinct products", False), ("mean_total_events", "total session events", False)]
    traits = [t for t in traits if t[0] in prof.columns]
    models = ordered_models(prof["model"].unique())
    fig, axes = plt.subplots(1, len(traits), figsize=(3.6 * len(traits), 3.6))
    x = np.arange(len(ERROR_GROUPS))
    for ax, (col, label, pct) in zip(np.atleast_1d(axes), traits):
        for i, m in enumerate(models):
            d = {r["group"]: r[col] for r in prof.filter(pl.col("model") == m).iter_rows(named=True)}
            ax.scatter(x + (i - (len(models) - 1) / 2) * 0.12, [d.get(g, np.nan) for g in ERROR_GROUPS], s=36,
                       color=COLOURS[m], edgecolor="white", linewidth=1, zorder=3, label=m)
        ax.set_xticks(x, [ERR_LABEL[g] for g in ERROR_GROUPS])
        ax.set_title(label, fontsize=10)
        ax.grid(axis="x", visible=False)
        if pct:
            ax.yaxis.set_major_formatter(PercentFormatter(1))
    np.atleast_1d(axes)[-1].legend(loc="upper left", bbox_to_anchor=(1.02, 1))
    fig.suptitle(f"Session traits by outcome (K={K0}, explained sample)", x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    save_figure(fig, cfg, "xai_error_traits")
    plt.show()
    display(prof.select("model", "group", "n", *[c for c in prof.columns if c.startswith("mean_")])
            .with_columns(pl.col(pl.Float64).round(3)).sort("model", "group"))

In [ ]:
if KS and focus:
    e = by_error[K0].filter(pl.col("model") == focus)
    top = feats_imp[K0].sort(focus, descending=True).head(12)["feature"].to_list()
    mat = np.array([[e.filter((pl.col("group") == g) & (pl.col("feature") == f))["mean_attribution"].sum()
                     for g in ERROR_GROUPS] for f in top])
    lim = np.abs(mat).max() or 1
    cmap = LinearSegmentedColormap.from_list("div", [PALETTE[1], "#f0efec", PALETTE[0]])
    fig, ax = plt.subplots(figsize=(7, 0.4 * len(top) + 1.6))
    im = ax.imshow(mat, cmap=cmap, vmin=-lim, vmax=lim, aspect="auto")
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            ax.text(j, i, f"{mat[i, j]:+.2f}", ha="center", va="center", fontsize=8)
    n = {r["group"]: r["n"] for r in e.unique("group").iter_rows(named=True)}
    ax.set_xticks(range(len(ERROR_GROUPS)), [f"{ERR_LABEL[g]}\n(n={n.get(g, 0)})" for g in ERROR_GROUPS])
    ax.set_yticks(range(len(top)), top)
    ax.grid(False)
    fig.colorbar(im, ax=ax, shrink=0.8, label="mean attribution (+ pushes towards 'buy')")
    ax.set_title(f"{focus}: what pushed each outcome (K={K0})")
    save_figure(fig, cfg, "xai_attribution_by_outcome")
    plt.show()

## 7. Performance by eventual session length

Test sessions grouped by their **total** length (analysis-only). If recall is low for
sessions that turn out long, the first K events miss intent that appears later — a
limit of early prediction rather than of the model.

In [ ]:
if KS:
    c = cohorts[K0]
    bins = c["group"].unique(maintain_order=True).to_list()
    bins = sorted(bins, key=lambda b: float(b.strip("[").split(",")[0]))
    models = ordered_models(c["model"].unique())
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharey=False)
    for ax, metric in zip(axes, ("recall", "precision", "pr_auc")):
        for m in models:
            d = {r["group"]: r[metric] for r in c.filter(pl.col("model") == m).iter_rows(named=True)}
            ax.plot(range(len(bins)), [d.get(b, np.nan) for b in bins], marker="o", markersize=5,
                    color=COLOURS[m], label=m)
        if metric == "pr_auc":
            base = {r["group"]: r["base_rate"] for r in c.unique("group").iter_rows(named=True)}
            ax.plot(range(len(bins)), [base.get(b, np.nan) for b in bins], color=INK_MUTED, linestyle="--",
                    linewidth=1, label="base rate")
        ax.set_xticks(range(len(bins)), bins, fontsize=8)
        ax.set_xlabel("total events in session")
        ax.set_title(metric, fontsize=10)
    axes[-1].legend(loc="upper left", bbox_to_anchor=(1.02, 1))
    fig.suptitle(f"Metrics by eventual session length (K={K0}, explained sample)", x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    save_figure(fig, cfg, "xai_length_cohorts")
    plt.show()
    display(c.with_columns(pl.col(pl.Float64).round(3)).sort("model", "group"))

## 8. Do all models miss the same buyers?

Model-agnostic, from the primary seed's **full** RQ1 test predictions at the default K —
so the GRU and the rule baseline are included. Jaccard overlap of the missed buyers
(false negatives) between models, then a profile of buyers that **every** model misses
against buyers that at least one model catches.

In [ ]:
overlap_ok = False
try:
    rq1 = latest_runs(cfg, "rq1_models")
    models_all = ordered_models(rq1.filter(pl.col("k") == K0)["model"].unique())
    preds = {m: primary_predictions(cfg, rq1.filter((pl.col("k") == K0) & (pl.col("model") == m)))
             for m in models_all}
    overlap_ok = bool(preds)
except RuntimeError as exc:
    print(f"[missing] RQ1 runs — {exc}")

if overlap_ok:
    ref = next(iter(preds.values()))
    assert all(p["user_session"].to_list() == ref["user_session"].to_list() for p in preds.values()), \
        "models were scored on different test sessions"
    y = ref["y_true"].to_numpy() == 1
    missed = {m: (p["y_pred"].to_numpy() == 0) & y for m, p in preds.items()}
    n = len(models_all)
    jac = np.eye(n)
    for i, a in enumerate(models_all):
        for j, b in enumerate(models_all):
            if i != j:
                union = (missed[a] | missed[b]).sum()
                jac[i, j] = (missed[a] & missed[b]).sum() / union if union else np.nan
    cmap = LinearSegmentedColormap.from_list("seq", ["#f6f9fe", PALETTE[0], "#0d366b"])
    fig, ax = plt.subplots(figsize=(1.0 * n + 2.5, 0.8 * n + 1.2))
    im = ax.imshow(jac, cmap=cmap, vmin=0, vmax=1)
    for i in range(n):
        for j in range(n):
            if i != j and np.isfinite(jac[i, j]):
                ax.text(j, i, f"{jac[i, j]:.2f}", ha="center", va="center", fontsize=8,
                        color="white" if jac[i, j] > 0.6 else "#0b0b0b")
    ax.set_xticks(range(n), models_all, rotation=40, ha="right")
    ax.set_yticks(range(n), [f"{m} (FN={missed[m].sum()})" for m in models_all])
    ax.grid(False)
    fig.colorbar(im, ax=ax, shrink=0.8, label="Jaccard overlap of missed buyers")
    ax.set_title(f"Missed buyers shared between models (K={K0}, test)")
    save_figure(fig, cfg, "xai_missed_overlap")
    plt.show()

In [ ]:
if overlap_ok:
    caught_by = np.sum([~missed[m] & y for m in models_all], axis=0)
    status = np.where(~y, "non-buyer", np.where(caught_by == 0, "buyer missed by all", "buyer caught by ≥1"))
    feats = load_feature_table(cfg, K0, "test").select(
        "user_session", "has_cart", "cart_count", "first_cart_position", "unique_products_count",
        "session_duration_sec", "last_event_type")
    prof = (pl.DataFrame({"user_session": ref["user_session"], "status": status})
            .join(feats, on="user_session", how="left")
            .group_by("status").agg(
                sessions=pl.len(),
                has_cart=pl.col("has_cart").cast(pl.Float64).mean(),
                cart_count=pl.col("cart_count").mean(),
                first_cart_position=pl.col("first_cart_position").mean(),
                distinct_products=pl.col("unique_products_count").mean(),
                median_duration_sec=pl.col("session_duration_sec").median(),
                last_event_cart=(pl.col("last_event_type") == "cart").mean(),
            ).sort("status"))
    display(prof.with_columns(pl.col(pl.Float64).round(3)))
    n_buyers = int(y.sum())
    n_all_missed = int(((caught_by == 0) & y).sum())
    print(f"{n_all_missed} of {n_buyers} test buyers ({n_all_missed / max(n_buyers, 1):.1%}) are missed by every model.")
    summary["buyers_missed_by_all"] = {"n": n_all_missed, "of": n_buyers}
    summary["missed_by_all_profile"] = prof.to_dicts()

## 9. Key findings

In [ ]:
lines = []
if "H5" in summary:
    lines.append(f"H5 (cart signals rank first): {summary['H5']['verdict']} — G2 top in {summary['H5']['cart_group_top']} (model, K) pairs.")
if "top_features" in summary:
    lines.append(f"Top features at K={K0} (mean share over models): " + ", ".join(
        f"{r['feature']} {r['mean_share']:.0%}" for r in summary["top_features"][:3]) + ".")
if "mean_rank_agreement" in summary:
    a = {r["level"]: r["mean_rho"] for r in summary["mean_rank_agreement"] if r["k"] == K0}
    lines.append("Mean rank agreement between models: " + ", ".join(f"{lvl} ρ = {v:.2f}" for lvl, v in a.items()) + ".")
if "buyers_missed_by_all" in summary:
    b = summary["buyers_missed_by_all"]
    lines.append(f"{b['n']} of {b['of']} test buyers are missed by every model (K={K0}).")
print("\n".join(f"- {line}" for line in lines) or "No RQ5 results yet.")

out = TABLES / f"rq5_explain_summary_{MODE}.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(summary, indent=2, default=float))
print(f"\nsaved {out}")

# Fill in the H5 row of the scoreboard written by notebook 03, if it exists.
board_path = TABLES / f"hypothesis_scoreboard_{MODE}.csv"
if board_path.exists() and "H5" in summary:
    board = pl.read_csv(board_path).with_columns(
        verdict=pl.when(pl.col("id") == "H5").then(pl.lit(summary["H5"]["verdict"])).otherwise(pl.col("verdict")),
        evidence=pl.when(pl.col("id") == "H5").then(pl.lit(f"G2 top in {summary['H5']['cart_group_top']} (model, K)"))
        .otherwise(pl.col("evidence")))
    board.write_csv(board_path)
    display(board)